In [ ]:
import os
import csv
import json
import requests
import pandas as pd
from pathlib import Path

print('Libraries imported successfully')

In [ ]:
# Set up paths
baseline_csv = Path('../../data/import/clean/all-briefings.csv')
output_dir = Path('../../data/export/ai_cleaned-up')
output_csv = output_dir / 'ollama.csv'

# Create output directory if needed
output_dir.mkdir(parents=True, exist_ok=True)

print(f'Loading baseline from: {baseline_csv}')
print(f'Output will be saved to: {output_csv}')

In [ ]:
# Load baseline videos
df = pd.read_csv(baseline_csv)
print(f'Loaded {len(df)} videos')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Ollama classification function
def classify_with_ollama(title, description=''):
    """Ask Ollama: Does BabyDog appear in this video? Answer only: True or False"""
    try:
        prompt = f"Does a baby dog (puppy/pup/young dog) appear in this video titled '{title}'?"
        if description:
            prompt += f" Description: {description}"
        prompt += "\n\nAnswer only 'True' or 'False':"
        
        response = requests.post(
            'http://localhost:11434/api/generate',
            json={
                'model': 'qwen2.5:4b',
                'prompt': prompt,
                'stream': False
            }
        )
        
        result = response.json()['response'].strip().lower()
        return 'True' in result or 'yes' in result
    except Exception as e:
        print(f'Error classifying \"{title}\": {e}')
        return None

print('Ollama classification function ready')

In [ ]:
# Classify all videos
results = []

for idx, row in df.iterrows():
    print(f'[{idx+1}/{len(df)}] Processing: {row[\"title\"][:50]}...')
    
    contains_babydog = classify_with_ollama(row['title'])
    
    result = {
        'video_id': row.get('video_id', ''),
        'title': row.get('title', ''),
        'date': row.get('date', ''),
        'youtube_link': row.get('youtube_link', ''),
        'contains_babydog': 'True' if contains_babydog else 'False',
        'interesting': '',
        'reviewer': 'Ollama',
        'note': ''
    }
    results.append(result)

print(f'\\nClassification complete: {len(results)} videos processed')

In [ ]:
# Save results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)

print(f'Saved to {output_csv}')
print(f'\\nSummary:')
print(f'Total: {len(results_df)}')
print(f'BabyDog found: {(results_df[\"contains_babydog\"] == \"True\").sum()}')
print(f'No BabyDog: {(results_df[\"contains_babydog\"] == \"False\").sum()}')
print(f'\\nFirst few rows:')
results_df.head()